# Машинное обучение, ФКН ВШЭ

## Практическое домашнее задание 2. Градиентный спуск своими руками

### Общая информация

Дата выдачи: 09.02.2026

Мягкий дедлайн: 23.02.2026 23:59 MSK

Жесткий дедлайн: 01.03.2026 23:59 MSK


### Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). **Максимально допустимая оценка за работу — 10 баллов + 0.5 за социальный бонус.**

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо считываемые диаграммы.

Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

**Устная проверка.** Для проверки понимания кода и выводов студент может быть приглашён на устную защиту. Оценка за задание может быть изменена после устной защиты. Если студент не может объяснить ключевые части решения и принятые решения, работа считается недобросовестной и оценивается в 0 баллов независимо от автотестов.

### Формат сдачи

Задания сдаются через систему Anytask. Инвайт можно найти на странице курса. Присылать необходимо ноутбук с выполненным заданием, а также файлы `descents.py` и `linear_regression.py`. Сам ноутбук называйте в формате **homework-practice-02-gd-Username.ipynb**, где Username - ваша фамилия.

Для удобства проверки самостоятельно посчитайте свою максимальную оценку (исходя из набора решенных задач) и укажите ниже.

**Оценка**: ...


### О задании

В данном задании необходимо реализовать обучение линейной регрессии с помощью различных модификаций градиентного спуска. В файле `descents.py` вам нужно будет реализовать несколько классов для различных вариаций градиентного спуска, а именно:
* `VanillaGradientDescent`
* `StochasticGradientDescent`
* `SAGDescent`
* `MomentumDescent`
* `Adam`


В файле `linear_regression.py` вам необходимо будет реализовать класс `CustomLinearRegression` для обучения линейной регрессии (и, разумеется, предсказания целевой переменной на основе обученной модели).


### Про предложенную архитектуру

Предложенная вам архитектура шаблонов написана по принципам SOLID: основная ее идея в том, что вы сможете использовать различные лоссы и оптимизаторы с одним и тем же кодом прочих классов, никак не изменяя и не переписывая методы классов, которые с оптимизаторами и лоссом взаимодействуют. Мы добиваемся этого при помощи выделения интерфейсов (в Python мы достигаем этого при помощи абстрактных классов, см дальше в заданиях) и выделения зон ответственности каждого класса.

Глобально в нашей архитектуре всего 4 интерфейса (некоторые из которых на самом деле сразу concrete классы), каждый из которых порождает одно семейство:
- `interfaces.LossFunction`
  - Классы, имплементирующие этот интерфейс, отвечают за одну конкретную функцию потерь, используемую при обучении, и всё, что меняется вместе с ней при её замене: подсчёт лосса, подсчет градиента и аналитическое решение (если есть, то добавляется в интерфейс соответствующим mixin-ом).
- `interfaces.LinearRegressionInterface`
  - Интерфейс обертки для модели линейной регрессии, контейнер, содержащий составные части (лосс-функцию и оптимизатор) и использующий их для выполнения содержательной работы.
- `interfaces.LearningRateSchedule`
 - Простенькое семейство расписаний, определяющих шаг обучения для каждой итерации
- `interfaces.AbstractOptimizer`.
  -  Классы, имплементирующие этот интерфейс, имплементируют конкретный алгоритм оптимизации и всё, что происходит в его процессе. Пользуются обёрткой линейной регрессии для доступа к данным и вызова расчетов, чтобы не зависеть напрямую от конкретных функций потерь и шедулеров шага обучения.

Концепция передачи маленьких объектов, отвечающих за свою маленькую зону ответственности внутрь более сложного объекта для выполнения ими составных частей работы называется Dependency Injection, и работает как раз за счет выделения зоны ответственности и опоры на интерфейс вместо реализации.

Посмотрите на код `linear_regression.CustomLinearRegression`: она принимает в себя объекты с интерфейсами `LossFunction` и `AbstractOptimizer`, а в `descents.BaseDescent` как уточнении интерфейса абстрактного оптимизатора до итеративных оптимизаторов видно, что он в свою очередь принимает при инициализации объект шедулера шага обучения.

Благодаря этому, код, который использует эти классы, может по очереди:
- Инициализировать нужный шедулер с нужными параметрами для задания шага обучения
- Иницализировать оптимизатор, задав ему нужные параметры процесса и передав готовый шедулер
- Инициализировать класс линейной регрессии нужной под задачу функцией потерь и уже готовым оптимизатором.
- Запустить процесс обучения

И на любом этапе можно использовать другой объект с подходящим интерфейсом, и всё будет работать!


**В ходе выполнения этой домашки вы наполните все эти семейства классов различными имплементациями и будете менять их на ходу как перчатки!**

Более подробно про наследование классов в Python можно прочитать здесь:
* Наследование: https://docs.python.org/3/tutorial/classes.html#inheritance
* Абстрактные классы: https://docs.python.org/3/library/abc.html



## Задание 1. Линейная регрессия  (1 балл)

### Градиент функции потерь MSE

На семинаре про [матрично-векторное дифференцирование](https://github.com/esokolov/ml-course-hse/blob/master/2025-fall/seminars/sem03-vector-diff.pdf) вы должны были обсуждать дифференцирование функции потерь MSE в матричном виде.

### Задание 1.0. Градиент MSE в матричном виде (0.3 балла).

Напомним, что функция потерь MSE записывается как:

$$
    Q(w) = \frac{1}{\ell} \sum \limits_{i = 1}^\ell (y_i - \langle x_i, w \rangle)^2 = \frac{1}{\ell} \| X w - y \|^2
$$

где $\ell$ – количество объектов в выборке, $X \in \mathbb{R}^{\ell \times d}$ – матрица "объект-признак", а $y \in \mathbb{R}^\ell$ – целевая переменная. Через $x_i$ обозначается $i$-ая строчка матрицы $X$, отвечающая за $i$-й объект выборки.

- **Выпишите ниже (подсмотрев в семинар или решив самостоятельно) градиент для функции потерь MSE в матричном виде.**

**Решение:**

Для начала перепишу функцию $$ Q(w) = \frac{1}{\ell} \sum \limits_{i = 1}^\ell (y_i - \langle x_i, w \rangle)^2 $$ в матричном виде:
$$
e = y - Xw = \begin{bmatrix}
y_1 - x_1^Tw\\
...\\
y_n - x_n^Tw
\end{bmatrix} = \begin{bmatrix}
e_1\\
...\\
e_n
\end{bmatrix}
$$

$$
e^Te = \sum \limits_{i = 1}^n (e_i)^2
$$

$$
(y - Xw)^T(y - Xw) = \sum \limits_{i = 1}^n (y_i - x_iw)^2
$$

$$
Q(w) = \frac{1}{\ell} (y - Xw)^T(y - Xw)
$$

$$
dQ = \frac{1}{\ell} d(y - Xw)^T(y - Xw) + \frac{1}{\ell} (y - Xw)^Td(y - Xw) = -\frac{1}{\ell} \begin{bmatrix}
dw^TX^T(y - Xw) + (y - Xw)^TXdw
\end{bmatrix} = -\frac{2}{\ell}(y - Xw)^TXdw
$$

Получается, что градиент MSE в матричном виде выглядит следующим образом:
$$
\nabla_w Q = \frac{2}{\ell}X^T(Xw - y)
$$


- **Имплементируйте методы `MSELoss.loss`, `MSELoss.gradient`**





### Задание 1.1 Аналитическое решение и `CustomLinearRegression` (0.7 балла)

Перед тем, как мы углубимся в итеративные методы оптимизации, давайте вспомним, что для ряда функций потерь существует и аналитическое решение. Давайте сперва вспомним, как оно выглядит для MSE.

- **Выведите формулу оптимальных $w$ в задаче минимизации MSE, и запишите её ниже.**

- **Имплементируйте подсчет этого решения в `MSELoss._plain_analytic_solution`**

$$\text{MSE} = \| X w - y \|^2$$
Приравняем производную к 0:
$$
\nabla_w Q = 0
$$
$$
\frac{2}{\ell}X^T(Xw - y) = 0
$$
Сократим $\frac{2}{\ell}$:
$$
X^TXw = X^Ty
$$
$$
w = (X^TX)^{-1}X^Ty
$$

**Вопрос**: Как мы помним, у аналитического решения есть минусы - какие?

**Ответ**:
1. Вычисление матрицы $X^TX$ имеет сложность $O(n^3)$, где $n$ - количество признаков, поэтому при большом количестве признаков вычисление становится очень медленным
2. Могут произойти проблемы при вычислении обратной матрицы $X^TX$, если, например, признаки линейно зависимы
3. Опять-таки, если матрица $X^TX$ близка к вырожеднной, то ошибки округления могут быть огромными

Теперь прокинем это решение в наш класс линейной регрессии, чтобы получше разобраться в архитектуре.


- **Допишите класс `descents.AnalyticSolutionOptimizer`**
- **Допишите класс `CustomLinearRegression`**
  - В нем на текущем этапе нужно имплементировать все методы: `fit` и `predict` вам понадобятся прямо сейчас, а `compute_gradients` и `compute_loss` в следующей части.

Помните, про разделение ответственности классов!

За контроль процесса обучения отвечает оптимизатор, а объект линейной регрессии по факту выступает точкой входа, контейнером для данных и способом доступа к вычислениям, на основе которых оптимизатор принимает решения (e.g. значение антиградиента в точке весов).

При этом сам по себе оптимизатор должен быть универсален, в нем никак не должны содержаться детали, связанные с конкретными функциями потерь, все необходимое от них он может получить через `self.model`.

Аналогично, класс линейной регрессии тоже должен быть универсальным и готовым к работе с любыми лоссами и оптимизаторами, исполняющими заявленный интерфейс. Здесь применена dependency injection, и вы должны грммотно ее поддержать в своей имплементации. Аналогично, все, что вам может быть нужно от функций потерь, вы можете получить при помощи обращения к переданному объекту.

In [ ]:
import numpy as np
from linear_regression import MSELoss, CustomLinearRegression, AnalyticSolutionOptimizer

num_objects = 100
dimension = 5

x = np.random.rand(num_objects, dimension)
y = np.random.rand(num_objects)

In [ ]:
from sklearn.metrics import mean_squared_error as mse
import sklearn

sklearn_linreg = sklearn.linear_model.LinearRegression(fit_intercept=False)
sklearn_linreg.fit(x, y)
print("Sklearn MSE", mse(sklearn_linreg.predict(x), y))

your_linreg = CustomLinearRegression(AnalyticSolutionOptimizer(), loss_function=MSELoss())
your_linreg.fit(x, y)
print("Your MSE", mse(your_linreg.predict(x), y))

assert abs(mse(your_linreg.predict(x), y) - mse(sklearn_linreg.predict(x), y)) < 1e-12, "Не повезло, попробуйте еще раз"

Давайте сделаем задание немного прикольнее и изменим одну из колонок. Как мы знаем, полная мультиколлинеарность запрещает нам пользоваться аналитическим решением, но `sklearn` по какой-то причине это обходит, хмм

In [ ]:
x[:, 3] = x[:, 2] + x[:, 4]

In [ ]:
sklearn_linreg = sklearn.linear_model.LinearRegression(fit_intercept=False)
sklearn_linreg.fit(x, y)
print("Sklearn MSE", mse(sklearn_linreg.predict(x), y))

your_linreg = CustomLinearRegression(AnalyticSolutionOptimizer(), loss_function=MSELoss())
your_linreg.fit(x, y)
print("Your MSE", mse(your_linreg.predict(x), y))

Ваша задача - понять, как можно сделать так, чтобы аналитическое решение работало всегда, вне зависимости от матрицы X. Как оказывается, это можно сделать, если воспользоваться SVD разложением. Для имплементации воспользуйтесь `scipy.sparse.linalg.svds`.

- Выведите через SVD формулу оптимальных $w$ в задаче минимизации MSE.

- Имплементируйте подсчет этого решения в `MSELoss._svd_analytic_solution`
    - Мир итерационных агоритмов причудлив. Если вы посмотрите на опции солверов svds, то увидите, что возможности вычислить точно все сингулярные числа вам не дают (propack рандомизированный и даст вам неточные ответы). Используйте стандартный солвер, выставьте максимальную доступную точность.

- Ответьте на **вопрос на засыпку**. Вообще говоря, в ряде случаев (например в нашем), даже такая неабсолютная на первый взгляд точность все равно позволяет получить точное решение задачи. Обоснуйте, почему? Как называется такой вид SVD? Какого минимального числа сингулярных чисел с вероятностью 1 будет достаточно в нашем случае для получения точного решения? Обоснуйте, почему.


$$\text{X} = \underset{n\times m}{\mathrm{U}} \ \underset{m\times m}{\mathrm{\Sigma}} \ \underset{m\times k}{\mathrm{V^T}}$$
Если $rank(X) = m$:
$$\nabla_w L(w) = \frac{2}{N}  X^T(Xw - y) = 0$$
$$X^TXw = X^Ty$$
$$(U\Sigma V^T)^T(U\Sigma V^T)w = V\Sigma U^Ty$$
$$V\Sigma^2V^Tw = V\Sigma U^Ty$$
$$w = V\Sigma^-1U^Ty$$
Если $r = rank(X) < m$:

Надо брать компактное SVD, иначе матрица $\Sigma$ необратима
$$X' = \underset{n\times r}{\mathrm{U}} \ \underset{r\times r}{\mathrm{\Sigma}} \ \underset{r\times k}{\mathrm{V^T}}$$
Аналогично получим:
$$w = V_r\Sigma_r^{-1}U_r^Ty$$

**Ответ**:

Для решения задачи минимизации MSE нам нужны только ненулевые сингулярные числа матрицы X. В нашем примере, когда мы сделали x[:, 3] = x[:, 2] + x[:, 4], матрица X стала вырожденной, но компактное SVD все равно нашло правильное решение, потому что нулевые сингулярные числа были проигнорированы.

Минимальное число сингулярных чисел, которое необходимо вычислить, равно рангу матрицы X (обозначим его r). Нулевые сингулярные числа не влияют на решение — они обнуляются в псевдообратной матрице. Поэтому для точного решения достаточно знать только $r$ ненулевых сингулярных чисел.

In [ ]:
sklearn_linreg = sklearn.linear_model.LinearRegression(fit_intercept=False)
sklearn_linreg.fit(x, y)
print("Sklearn MSE", mse(sklearn_linreg.predict(x), y))


your_linreg =  CustomLinearRegression(AnalyticSolutionOptimizer(),
                                      loss_function=MSELoss(analytic_solution_func=MSELoss._svd_analytic_solution))
your_linreg.fit(x, y)

print("Your MSE", mse(your_linreg.predict(x), y))

assert abs(mse(your_linreg.predict(x), y) - mse(sklearn_linreg.predict(x), y)) < 1e-12, "Не повезло, попробуйте еще раз"

## Задание 2. Реализация градиентного спуска (4 балла)

В этом задании вам предстоит написать собственные реализации различных подходов к градиентному спуску с опорой на подготовленные шаблоны в файле `descents.py`. При помощи них мы будем искать итеративные решения, подавая оптимизаторы внутрь нашей `CustomLinearRegression`.

### Напоминание про градиентный спуск

Основное свойство антиградиента &ndash; он указывает в сторону *наискорейшего* убывания функции в данной точке. Соответственно, будет логично стартовать из некоторой точки, сдвинуться в сторону антиградиента,
пересчитать антиградиент и снова сдвинуться в его сторону и т.д. Запишем это более формально.

Пусть $w_0$ &ndash; начальный набор параметров (например, нулевой или сгенерированный из некоторого
случайного распределения). Тогда ванильный градиентный спуск состоит в повторении следующих шагов до сходимости:

$$
    w_{k + 1} = w_{k} - \eta_{k} \nabla_{w} Q(w_{k}).
$$

Здесь $\eta_{k}$ обозначает длину шага на $k$-ой итерации (learning rate), а $Q(w)$ - функцию потерь (loss function).

Градиент для MSE вы уже нашли выше

### Задание 2.0. Learning Rate Schedules (0.2 балла)

Обратите внимание на **абстрактный** класс `LearningRateSchedule` в файле `descents.py`. С помощью его имплементаций мы на каждой итерации градиентного спуска будем получать соответствующий `learning_rate` $\eta_k$.

В файле уже реализован класс `ConstantLR`, который на каждой итерации возвращает один и тот же заранее заданный шаг. **Ваша задача в этом пункте – реализовать `TimeDecayLR`**, который мы будем использовать для обучения линейной регрессии. Формула очередного шага должна выглядеть следующим образом:
$$
    \eta_{k} = \lambda \left(\dfrac{s_0}{s_0 + k}\right)^p
$$

На практике достаточно настроить параметр $\lambda$, а остальным выставить параметры по умолчанию: $s_0 = 1, \, p = 0.5.$

### Задание 2.1. Родительский класс BaseDescent (1 балл).


Внимательно изучите устройство класса `BaseDescent`. У него есть один непомеченный абстрактным метод, который ему как частичному наследнику абстрактного класса нужно имплементировать - это `optimize`. В
этом методе необходимо имплементировать основной цикл обучения, и далее его будут переиспользовать все его наследники.

- **Допишите метод `BaseDescent.optimize`**


Для этого и всех дальнейших заданий необходимо соблюдать следующие условия:

* **Все вычисления должны быть векторизованы;**
* Циклы средствами python допускаются только для итераций градиентного спуска;
* В качестве критерия останова необходимо использовать (одновременно):
    * Квадрат евклидовой нормы разности весов на двух соседних итерациях меньше `tolerance`;
    * Разность весов содержит наны;
    * Достижение максимального числа итераций `max_iter`.
* Будем считать, что все данные, которые поступают на вход имеют столбец единичек последним столбцом;
* Веса модели надо обновлять внутри функции `_update_weights`, она неспроста так называется
* Чтобы проследить за сходимостью оптимизационного процесса будем использовать `CustomLinearRegression.loss_history`, в нём будем хранить *значения функции потерь до каждого шага, начиная с нулевого* (до первого шага по антиградиенту) и *значение функции потерь после оптимизации*.


Обратите внимание, что метод `_update_weights` всё ещё является абстрактным - его все ещё должны будут имплементировать дальнейшие наследники; фактически, только способом обновления весов они и отличаются. Она должна должна обновлять веса модели `self.model.w`, а также возвращать величину обновления $w_{k + 1} - w_k$.

Также обратите внимание на атрибут `self.iteration`, отвечающий за номер итерации алгоритма спуска. Как раз с помощью него (и `self.lr_schedule`) мы и будем получать `learning_rate` на соответствующей итерации алгоритма.

**Обратите внимание**

*да, в третий раз*

Все реализуемые вами классы спуска в задании - это *универсальные* оптимизаторы. Они не должны считать градиенты конкретной функции потерь внутри себя.

Для вычисления градиента они всегда обращаются к модели, с которой работают:

```
gradient = self.model.compute_gradients(X_batch, y_batch)
```

### Задание 2.2. Полный градиентный спуск VanillaGradientDescent (0.6 балла).

Реализуйте полный градиентный спуск заполнив пропуски в классе `VanillaGradientDescent` в файле `descents.py`. Напомним, что шаг классического градиентного спуска выглядит следующим образом:

$$
    w_{k + 1} = w_{k} - \eta_{k} \nabla_{w} Q(w_{k}).
$$

**Важно**: Здесь и далее функция `_update_weights` должна возвращать разницу между $w_{k + 1}$ и $w_{k}$: $\quad w_{k + 1} - w_{k} = -\eta_{k} \nabla_{w} Q(w_{k})$. Кроме того, соответственно своему названию, она должна обновлять веса модели `model.w`.

### Напоминание про SGD (стохастических градиентный спуск)

Как правило, в задачах машинного обучения функционал $Q(w)$ представим в виде суммы $\ell$ функций:

$$
    Q(w)
    =
    \frac{1}{\ell}
    \sum_{i = 1}^{\ell}
        q_i(w).
$$

В нашем домашнем задании отдельные функции $q_i(w)$ соответствуют ошибкам на отдельных объектах.

Проблема метода градиентного спуска состоит в том, что на каждом шаге необходимо вычислять градиент всей суммы (будем его называть полным градиентом):

$$
    \nabla_w Q(w)
    =
    \frac{1}{\ell}
    \sum_{i = 1}^{\ell}
        \nabla_w q_i(w).
$$

Это может быть очень трудоёмко при больших размерах выборки. В то же время точное вычисление градиента может быть не так уж необходимо &ndash; как правило, мы делаем не очень большие шаги в сторону антиградиента, и наличие в нём неточностей не должно сильно сказаться на общей траектории.

Оценить градиент суммы функций можно средним градиентов случайно взятого подмножества функций:

$$
    \nabla_{w} Q(w_{k}) \approx \dfrac{1}{|B|}\sum\limits_{i \in B}\nabla_{w} q_{i}(w_{k}),
$$
где $B$ - это случайно выбранное подмножество индексов, обычно называемое **батчом**.

Оценка $\frac{1}{|B|} \sum \limits_{i \in B} \nabla_w q_i(w_k)$ называется **стохастическим градиентом** функции потерь, а получившийся метод называют методом **стохастического градиентного спуска** или просто SGD.

### Задание 2.3. Стохастический градиентный спуск StochasticGradientDescent (0.7 балла).

Реализуйте стохастический градиентный спуск, заполнив пропуски в классе `StochasticGradientDescent`. Для оценки градиента используйте формулу выше (среднее градиентов случайно выбранного батча объектов). Шаг оптимизации:

$$
    w_{k + 1} = w_{k} - \eta_{k} \dfrac{1}{|B|}\sum\limits_{i \in B}\nabla_{w} q_{i}(w_{k}).
$$

Размер батча будет являться **гиперпараметром** метода и передаваться в конструктор класса `__init__(...)`. Семплировать индексы батча объектов $B$ можно с повторениями (через np.random.randint) - это допустимо и даёт несмещённую оценку градиента. По желанию можно без повторений (np.random.choice(..., replace=False) или через пермутацию по эпохам).

### Задание 2.4 Stochastic Average Gradient (0.6 балла)

Держим память последних индивидуальных градиентов $g_i$ по всем объектам и их среднее $\bar g = \frac{1}{\ell}\sum_i g_i$. На каждом шаге выбираем индексы $j$ (мини-батч), заново считаем $g_j^{new}(w_k)$, обновляем среднее:
$$
\bar g \leftarrow \bar g + \frac{1}{\ell}\bigl(g_j^{new} - g_j^{old}\bigr),\qquad
w_{k+1} = w_k - \eta_k \bar g.
$$
Инициализация: $g_i=0 \Rightarrow \bar g=0$.

Так получаем шаг почти как у полного градиента, но считаем градиент лишь на нескольких объектах за итерацию.

Реализуйте класс `SAGDescent` в `descents.py` с хранением `grad_memory` и `avg_grad`. Подсказка: чтобы получить пер-объектный градиент, можно вызывать `compute_gradients` на срезе из одного объекта `X[j:j+1]` или на фильтрованной индексами матрице для батча.

**Имейте в виду, что SAG достаточно капризный**: для его сходимости (и ее скорости) достаточно важен размер батча. Для вас установлено дефолтное значение, но на реальных данных его может быть недостаточно. В сравнениях методов ниже вам может понадобится подобрать значение размера батча, чтобы раскрыть потенциал метода. То ж касается и SGD, но в меньшей степени.

### Напоминание про метод инерции (или метод моментов)

Может оказаться, что направление антиградиента сильно меняется от шага к шагу. Например, если линии уровня функционала сильно вытянуты, то из-за ортогональности градиента линиям уровня он будет менять направление на почти противоположное на каждом шаге. Такие осцилляции будут вносить сильный шум в движение, и процесс оптимизации займёт много итераций. Чтобы избежать этого, можно усреднять векторы антиградиента с нескольких предыдущих шагов &ndash; в этом случае шум уменьшится, и такой средний вектор будет указывать в сторону общего направления движения. Введём для этого вектор инерции:

\begin{align}
    &h_0 = 0, \\
    &h_{k + 1} = \alpha h_{k} + \eta_k \nabla_w Q(w_{k})
\end{align}

Здесь $\alpha$ &ndash; параметр метода, определяющей скорость затухания градиентов с предыдущих шагов. Разумеется, вместо вектора градиента может быть использована его аппроксимация (например, в случае **стохастического градиентного спуска**). Чтобы сделать шаг градиентного спуска, просто сдвинем предыдущую точку на вектор инерции:

$$
    w_{k + 1} = w_{k} - h_{k + 1}.
$$

Заметим, что если по какой-то координате градиент постоянно меняет знак, то в результате усреднения градиентов в векторе инерции эта координата окажется близкой к нулю. Если же по координате знак градиента всегда одинаковый, то величина соответствующей координаты в векторе инерции будет большой, и мы будем делать большие шаги в соответствующем направлении.

### Задание 2.5 Метод Momentum - MomentumDescent (0.5 балла).

Реализуйте градиентный спуск с методом инерции заполнив пропуски в классе `MomentumDescent`. Шаг оптимизации:

\begin{align}
    &h_0 = 0, \\
    &h_{k + 1} = \alpha h_{k} + \eta_k \nabla_w Q(w_{k}) \\
    &w_{k + 1} = w_{k} - h_{k + 1}.
\end{align}

$\alpha$ являеться гиперпараметром метода, однако в данном домашнем задании мы зафиксируем её за вас $\alpha = 0.9$.

### Напоминание про AdaGrad, RMSprop и Adam

Градиентный спуск очень чувствителен к выбору длины шага. Если шаг большой, то есть риск, что мы будем перескакивать через точку минимума; если же шаг маленький, то для нахождения минимума потребуется много итераций. При этом нет способов заранее определить правильный размер шага &ndash; к тому же, схемы с постепенным уменьшением шага по мере итераций могут тоже плохо работать.

В методе AdaGrad предлагается сделать свою длину шага для каждой компоненты вектора параметров. Идея проста: мы будем "копить" сумму квадратов градиентов и делить очередной градиент на корень из этой суммы. Таким образом, обновление весов с большими градиентами будет тормозиться, а с маленькими наоборот получать большие шаги. Формула обновлени будет выглядить так:

\begin{align}
    &G_{kj} = G_{k-1,j} + (\nabla_w Q(w_{k - 1}))_j^2; \\
    &w_{jk} = w_{j,k-1} - \frac{\eta_t}{\sqrt{G_{kj}} + \varepsilon} (\nabla_w Q(w_{k - 1}))_j.
\end{align}

Здесь $\varepsilon$ небольшая константа, которая предотвращает деление на ноль.

В данном методе можно зафиксировать длину шага (например, $\eta_k = 0.01$) и не подбирать её в процессе обучения **(обратите внимание, что в данном домашнем задании длина шага не фиксируется)**. Отметим, что данный метод подходит для разреженных задач, в которых у каждого объекта большинство признаков равны нулю. Для признаков, у которых ненулевые значения встречаются редко, будут делаться большие шаги; если же какой-то признак часто является ненулевым, то шаги по нему будут небольшими.

У метода AdaGrad есть большой недостаток: переменная $G_{kj}$ монотонно растёт, из-за чего шаги становятся всё медленнее и могут остановиться ещё до того, как достигнут минимум функционала. Проблема решается в методе RMSprop, где используется экспоненциальное затухание градиентов:

$$
    G_{kj} = \alpha G_{k-1,j} + (1 - \alpha) (\nabla_w Q(w^{(k-1)}))_j^2.
$$

В этом случае размер шага по координате зависит в основном от того, насколько
быстро мы двигались по ней на последних итерациях.

Можно объединить идеи описанных выше методов: накапливать градиенты со всех прошлых шагов для
избежания осцилляций (метод инерции), а также делать адаптивную длину шага по каждому параметру (`RMSProp`). Таким образом, мы получим метод `Adam` с той лишь разницей, что в методе `Adam` дополнительно делается нормировка накопленных градиентов и квадратов градиентов для устранения смещения.

### Задание 2.6. Метод Adam (Adaptive Moment Estimation) (0.4 балла).

Реализуйте градиентный спуск с методом Adam, заполнив пропуски в классе `Adam`. Шаг оптимизации:

\begin{align}
    &m_0 = 0, \quad v_0 = 0; \\ \\
    &m_{k + 1} = \beta_1 m_k + (1 - \beta_1) \nabla_w Q(w_{k}); \\ \\
    &v_{k + 1} = \beta_2 v_k + (1 - \beta_2) \left(\nabla_w Q(w_{k})\right)^2; \\ \\
    &\widehat{m}_{k} = \dfrac{m_k}{1 - \beta_1^{k}}, \quad \widehat{v}_{k} = \dfrac{v_k}{1 - \beta_2^{k}}; \\ \\
    &w_{k + 1} = w_{k} - \dfrac{\eta_k}{\sqrt{\widehat{v}_{k + 1}} + \varepsilon} \widehat{m}_{k + 1}.
\end{align}

$\beta_1 = 0.9, \beta_2 = 0.999$ и $\varepsilon = 10^{-8}$ будут зафиксированы за вас.

## Задание 3. Проверка кода (0 баллов)

Данная секция нужна для того, чтобы убедиться в правильности реализации методов спуска и класса `CustomLinearRegression`. В начале мы сделаем небольшую локальную проверку на "адекватность" и "запускаемость" ваших моделей, после чего уже можно будет делать посылки в Яндекс Контест.

In [ ]:
%load_ext autoreload

In [ ]:
#%autoreload 2

from descents import (
    VanillaGradientDescent,
    StochasticGradientDescent,
    SAGDescent,
    MomentumDescent,
    Adam
)

from linear_regression import CustomLinearRegression

In [ ]:
num_objects = 100
dimension = 5

x = np.random.rand(num_objects, dimension)
y = np.random.rand(num_objects)

Проверяем код на запускаемость.

In [ ]:
descent_models = [
   VanillaGradientDescent,
   StochasticGradientDescent,
   SAGDescent,
   MomentumDescent,
   Adam
]

max_iter = 10
tolerance = 0
num_objects = 100
dimension = 5

for descent_model in descent_models:
   optimizer = descent_model(tolerance=tolerance, max_iter=max_iter)
   model = CustomLinearRegression(optimizer=optimizer)
   model.fit(x, y)
   assert len(model.loss_history) == max_iter + 1, "Loss history failed"
   y_pred = model.predict(x)
   assert y_pred.shape == y.shape, "Prediction shape does not match target variable"


Если ваше решение прошло все тесты локально, то теперь пришло время протестировать его в [Яндекс Контесте](https://new.contest.yandex.ru/contests/90083/start).

Для каждой задачи из контеста вставьте ID успешной посылки и ваш ник (почту):

* **Ник/почта**: ekaterinashelocheva


* **VanillaGradientDescent**: 157166575


* **StochasticGradientDescent**: 157166598


* **SAGDescent**: 157166605


* **MomentumDescent**: 157166619


* **Adam**: 157166637


* **LinearRegression**: 157166702

## Задание 4. Работа с данными (1 балл)

Мы будем использовать датасет объявлений по продаже машин на немецком Ebay. В задаче предсказания целевой переменной для нас будет являться цена.

* Постройте график распределения целевой переменной в данных, подумайте, нужно ли заменить её на логарифм. Присутствуют ли выбросы в данных с аномальной ценой? Если да, то удалите их из данных.

* Проведите исследование данных:
    * Проанализируйте тип столбцов, постройте графики зависимости целевой переменной от признака, распределения значений признака;
    * Подумайте (и напишите): какие признаки могут быть полезными на основе этих графиков, обработайте выбросы;
    * Подумайте (и напишите): какие трансформации признаков из известных вам будет уместно применить;
    * Разделите полезные признаки на категориальные, вещественные и те, которые не надо предобрабатывать.
* Разделите данные на обучающую, валидационную и тестовую выборки в отношении 8:1:1.

Видно, что после логарифмирования распределение становится более симметричным и похожим на нормальное

In [ ]:
import sys
!{sys.executable} -m pip install pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd  # при желании, можете заменить на polars/pyspark или что угодно, что вам нравится

import matplotlib.pyplot as plt
import seaborn as sns

from descents import (
    ConstantLR, TimeDecayLR,
    VanillaGradientDescent, StochasticGradientDescent,
    MomentumDescent, Adam, SAGDescent
)
from linear_regression import CustomLinearRegression

sns.set(style='darkgrid')

In [ ]:
data = pd.read_csv('autos.csv')  # разумеется, если вы используете не pandas, это надо поменять

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(data['price'], bins="auto", kde = True)
plt.title('Распределение цены')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(np.log1p(data['price']), bins=60, kde = True)
plt.title('Логарифм цены')
plt.show()

На первом графике распределение цены сильно скошено вправо, а большинство значений сконцентрированы в левой части. После логарифмирования распределение становится более симметричным и похожим на нормальное.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=np.log(data['price']))
plt.title('Box plot логарифма price')
plt.xlabel('log(price)')
plt.show()

In [ ]:
log_price = np.log(data['price'])
#Значение подобрала вручную, глядя на графики до и после удаления выбросов
data_filtered = data[log_price > 5.1]
data = data_filtered

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=np.log(data['price']))
plt.title('Box plot логарифма price после удаления выбросов')
plt.xlabel('log(price)')
plt.show()

In [ ]:
x = data.shape
y = data.shape
print(x, y)

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=np.log(data['powerPS']))
plt.title('Box plot логарифма powerPS')
plt.xlabel('log(powerPS)')
plt.show()

In [ ]:
log_price = np.log(data['powerPS'])
#Значение подобрала вручную, глядя на графики до и после удаления выбросов
data_filtered = data[log_price < 5.94]
data = data_filtered

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=np.log(data['powerPS']))
plt.title('Box plot логарифма powerPS после удаления выбросов')
plt.xlabel('log(powerPS)')
plt.show()

In [ ]:
x = data.shape
y = data.shape
print(x, y)

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=data['autoAgeMonths'])
plt.title('Box plot autoAgeMonths')
plt.xlabel('autoAgeMonths')
plt.show()

In [ ]:
log_price = data['autoAgeMonths']
#Значение подобрала вручную, глядя на графики до и после удаления выбросов
data_filtered = data[log_price < 310]
data = data_filtered

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=data['autoAgeMonths'])
plt.title('Box plot autoAgeMonths после удаления выбросов')
plt.xlabel('autoAgeMonths')
plt.show()

In [ ]:
x = data.shape
y = data.shape
print(x, y)

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=data['kilometer'])
plt.title('Box plot kilometer')
plt.xlabel('kilometer')
plt.show()

In [ ]:
log_price = data['kilometer']
#Значение подобрала вручную, глядя на графики до и после удаления выбросов
data_filtered = data[log_price > 20000]
data = data_filtered

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=data['kilometer'])
plt.title('Box plot kilometer')
plt.xlabel('kilometer')
plt.show()

In [ ]:
x = data.shape
y = data.shape
print(x, y)

In [ ]:
data.head()

Колонки в данных:

* `brand` - название бренда автомобиля
* `model` - название модели автомобиля
* `vehicleType` - тип транспортного средства
* `gearbox` - тип трансмисcии
* `fuelType` - какой вид топлива использует автомобиль
* `notRepairedDamage` - есть ли в автомобиле неисправность, которая еще не устранена
* `powerPS` - мощность автомобиля в PS (метрическая лошадиная сила)
* `kilometer` - сколько километров проехал автомобиль, пробег
* `autoAgeMonths` - возраст автомобиля в месяцах


* `price` - цена, указанная в объявлении о продаже автомобиля (целевая переменная)

Разделите признаки на категориальные, числовые и ... все остальное

In [ ]:
categorical = ["brand", "model", "vehicleType", "gearbox", "fuelType", "notRepairedDamage"]
numeric = ["powerPS", "kilometer", "autoAgeMonths"]
other = []

# YOUR CODE (EDA) HERE ٩(⁎❛ᴗ❛⁎)۶


In [ ]:
num_cols = ["powerPS", "kilometer", "autoAgeMonths"]

for c in num_cols:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=data[c], y=data["price"], alpha=0.15)
    plt.title(f"price vs {c}")
    plt.show()

1. Наблюдается прямая корреляция между мощностью двигателя (powerPS) и стоимостью автомобиля: более мощные автомобили характеризуются более высокой ценой.
2. Пробег (kilometer) имеет обратную зависимость с ценой: автомобили с меньшим пробегом стоят дороже.
3. Возраст автомобиля (autoAgeMonths) также обратно пропорционален цене: более старые автомобили имеют меньшую стоимость.

Трансформации:

1. Для целевой переменной price целесообразно применить логарифмическое преобразование для нормализации распределения, либо использовать клиппинг по процентилям для удаления экстремальных значений.
2. Признак kilometer не требует дополнительных преобразований, так как демонстрирует устойчивую монотонную зависимость.
3. Признак autoAgeMonths также не нуждается в трансформации, поскольку характеризуется четкой монотонной убывающей зависимостью с целевой переменной.

In [ ]:
category_cols = ["brand", "vehicleType", "gearbox", "fuelType", "notRepairedDamage", "model"]

for c in category_cols:
    top = data[c].value_counts().head(10).index

    tmp = data[data[c].isin(top)]

    plt.figure(figsize=(8,4))
    sns.barplot(data=tmp, x=c, y="price", estimator="mean", errorbar=None)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"sum price by {c}")
    plt.show()

Анализ по брендам: 
1. Наибольшую среднюю стоимость демонстрируют премиальные бренды (BMW, Audi, Mercedes-Benz). Примечательно, что Volkswagen, несмотря на более низкую среднюю цену, имеет сопоставимый с премиальными брендами суммарный объем продаж.
2. Тип кузова: Наблюдается значимая зависимость цены от типа кузова - седаны и кроссоверы характеризуются более высокой стоимостью по сравнению с другими типами.
3. Тип трансмиссии: Автомобили с автоматической коробкой передач в среднем стоят примерно в два раза дороже, чем с механической. Это может быть связано как с премиальностью сегмента, так и с тем, что в категории с механической трансмиссией преобладают более бюджетные модели.
4. Наличие повреждений: Автомобили с неисправностями (notRepairedDamage = ja) имеют существенно более низкую рыночную стоимость.

Трансформации признаков:

В признаке fuelType присутствуют редкие категории (электро, гибрид), которые характеризуются высокой стоимостью, но малым количеством наблюдений. Это не критично для модели, однако следует учитывать дисбаланс классов - подавляющее большинство объектов приходится на бензиновые и дизельные двигатели.

In [ ]:
category_cols = ["brand", "vehicleType", "gearbox", "fuelType", "notRepairedDamage", "model"]

for c in category_cols:
    top = data[c].value_counts().head(10).index

    tmp = data[data[c].isin(top)]

    plt.figure(figsize=(8,4))
    sns.barplot(data=tmp, x=c, y="price", estimator="sum", errorbar=None)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"sum price by {c}")
    plt.show()

1. Наблюдается, что автомобили с гибридным типом топлива характеризуются наиболее высокой стоимостью, однако их доля на рынке крайне мала.
2. Преобладающую часть выборки составляют автомобили без повреждений (notRepairedDamage = nein), которые ожидаемо имеют более высокую цену по сравнению с битыми автомобилями.

Добавляем в данные единичную колонку `bias`, чтобы не делать отдельные параметр $b$ для свободного члена модели.

In [ ]:
data['price'] = np.log1p(data['price'])

data['bias'] = 1
other += ['bias']

x = data[categorical + numeric + other]
y = data['price']

Теперь вам необходимо разбить данные на обучающую, тестовую и валидационную выборки.

In [ ]:
n = len(data)
idx = np.random.permutation(n)

n_train = int(0.8 * n)
n_test = int(0.1 * n)

train_idx = idx[:n_train]
test_idx = idx[n_train:n_train + n_test]
val_idx = idx[n_train + n_test:]


X_train = x.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)

X_val = x.iloc[val_idx].reset_index(drop=True)
y_val = y.iloc[val_idx].reset_index(drop=True)

X_test = x.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

А также сделаем базовую обработку данных, а именно:
* Применим `OneHotEncoding` к категориальным признакам
* Стандартизуем численные признаки с помощью `StandardScaler`
* Остальные признаки трогать не будем, т.к. с ними непонятно что делать

> А почему мы сначала делим данные, а только потом применяем обработку данных? Энкодеры и скейлеры используют информацию о данных: если сделать fit на всем датасете до split, это будет утечка: статистики из val/test попадут в обучение.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler


column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('scaling', StandardScaler(), numeric),
    ('other',  'passthrough', other)
])

X_train = column_transformer.fit_transform(X_train)
X_val = column_transformer.transform(X_val)
X_test = column_transformer.transform(X_test)

## Задание 5. Сравнение методов градиентного спуска (1.5 балла)

В этом задании вам предстоит сравнить методы градиентного спуска на подготовленных вами данных из предыдущего задания.

### Задание 5.1. Подбор оптимальной длины шага (0.75 балла)

Подберите по валидационной выборке наилучшую длину шага $\lambda$ для каждого метода с точки зрения ошибки. Для этого сделайте перебор по логарифмической сетке. Для каждого метода посчитайте ошибку на обучающей и тестовой выборках, посчитайте качество по метрике $R^2$, сохраните количество итераций до сходимости.

Все параметры кроме `lambda_` стоит выставить равным значениям по умолчанию.

In [ ]:
if hasattr(X_train, 'toarray'):
    X_train_np = X_train.toarray()
else:
    X_train_np = np.array(X_train)

if hasattr(X_val, 'toarray'):
    X_val_np = X_val.toarray()
else:
    X_val_np = np.array(X_val)

if hasattr(X_test, 'toarray'):
    X_test_np = X_test.toarray()
else:
    X_test_np = np.array(X_test)

print(f"Форма X_train: {X_train_np.shape}")
print(f"Форма X_val: {X_val_np.shape}")
print(f"Форма X_test: {X_test_np.shape}")

if hasattr(y_train, 'values'):
    y_train_np = y_train.values
else:
    y_train_np = np.array(y_train)

if hasattr(y_val, 'values'):
    y_val_np = y_val.values
else:
    y_val_np = np.array(y_val)

if hasattr(y_test, 'values'):
    y_test_np = y_test.values
else:
    y_test_np = np.array(y_test)

print(f"Форма y_train: {y_train_np.shape}")
print(f"Форма y_val: {y_val_np.shape}")
print(f"Форма y_test: {y_test_np.shape}")

In [ ]:
from sklearn.metrics import r2_score

from descents import (
    ConstantLR,
    TimeDecayLR,
    VanillaGradientDescent,
    StochasticGradientDescent,
    SAGDescent,
    MomentumDescent,
    Adam,
)

from linear_regression import (
    MSELoss, 
    L2Regularization,
    CustomLinearRegression, 
)

model_name = {
    VanillaGradientDescent: "VanillaGradientDescent",
    StochasticGradientDescent: "StochasticGradientDescent",
    SAGDescent: "SAGDescent",
    MomentumDescent: "MomentumDescent",
    Adam: "Adam"
}

best_lambda = {
    VanillaGradientDescent: 0,
    StochasticGradientDescent: 0,
    SAGDescent: 0,
    MomentumDescent: 0,
    Adam: 0
}

print(f"{'Model':<25} | {'LR':>12} | {'Iterations':>10} | {'Val Loss':>12} | {'Train Loss':>12} | {'r2_val':>10}")
print("-" * 140)

best_result = {}
min_loss = 1e9
tolerance = 1e-5
max_iter = 1000

batch_size = 4096 #batch подбирала вручную
log_grid = np.logspace(-4, 1, 30)

best_result_sagd = {}
best_result_sagd['SAGDescent'] = {
    "best_lr": None,
    "iterations": None,
    "train_loss": np.inf,
    "val_loss": np.inf,
    "r2_val": None,
    "batch_size": batch_size
}

for lr in log_grid:
    optimizer = SAGDescent(
        tolerance=tolerance, 
        max_iter=max_iter, 
        batch_size=batch_size, 
        lr_schedule=ConstantLR(lr)
    )
        
    model = CustomLinearRegression(optimizer=optimizer)
    model.fit(X_train, y_train)
    
    train_loss = model.compute_loss(X_train, y_train)
    test_loss = model.compute_loss(X_test, y_test)
    val_loss = model.compute_loss(X_val, y_val)
    
    if not (np.isfinite(train_loss) and np.isfinite(val_loss) and np.isfinite(test_loss)):
        continue

    r2_val = r2_score(y_val, model.predict(X_val))
    iterations = optimizer.iteration 
    
    print(f"{model_name[SAGDescent]:<25} | {lr:>12.6f} | {iterations:>10d} | {val_loss:>10e} | {train_loss:>10e} | {r2_val:>10e}")
    
    if min_loss > val_loss:
        best_lambda[SAGDescent] = lr
        min_loss = val_loss
        best_result_sagd['SAGDescent'].update({
            "best_lr": lr,
            "iterations": iterations,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "r2_val": r2_val,
            "batch_size": batch_size
        })

print()
print() 
res = best_result_sagd['SAGDescent']
print(f"BEST for {model_name[SAGDescent]}: lr={res['best_lr']:.6f}, iters={res['iterations']}, "
      f"val_mse={res['val_loss']:.2f}, r2_val={res['r2_val']:.4f}")
print()
print()
print("-" * 140)

In [ ]:
#Тут я не беру SAGD потому что я посчитала его выше
model_name_no = {
    VanillaGradientDescent: "VanillaGradientDescent",
   StochasticGradientDescent: "StochasticGradientDescent",
   MomentumDescent: "MomentumDescent",
   Adam: "Adam"}


log_grid = np.logspace(-4, 1, 30)

print(f"{'Model':<25} | {'LR':>12} | {'Iterations':>10} | {'Val Loss':>12} | {'Train Loss':>12} | {'r2_val':>10}")
print("-" * 140)

best_result = {}

tolerance = 1e-5
max_iter = 1000

for modelka in model_name_no:
    min_loss = 1e9
    best_result[modelka] = {
        "best_lr": None,
        "iterations": None,
        "train_loss": np.inf,
        "val_loss": np.inf,
        "r2_val": None,
        "batch_size" : np.inf
    }
    for lr in log_grid:
        
        optimizer = modelka(tolerance = tolerance, max_iter = max_iter, lr_schedule = ConstantLR(lr))
            
        model = CustomLinearRegression(optimizer=optimizer)
        model.fit(X_train, y_train)
        
        train_loss = model.compute_loss(X_train, y_train)
        test_loss = model.compute_loss(X_test, y_test)
        val_loss = model.compute_loss(X_val, y_val)
        
        if not (np.isfinite(train_loss) and np.isfinite(val_loss) and np.isfinite(test_loss)):
            continue

        r2_val = r2_score(y_val, model.predict(X_val))
        
        

        iterations = optimizer.iteration 
        print(f"{model_name_no[modelka]:<25} | {lr:>12.6f} | {iterations:>10d} | {val_loss:>10e} | {train_loss:>10e} | {r2_val:>10e}")
        if min_loss > val_loss:
            min_loss = val_loss
            best_result[modelka].update({
                "best_lr": lr,
                "iterations": iterations,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "r2_val": r2_val,
                "batch_size": np.inf
            })

            
    print()
    print()
    res = best_result[modelka]
    print(f"BEST for {model_name_no[modelka]}: lr={res['best_lr']:.6f}, iters={res['iterations']}, "
                f"val_mse={res['val_loss']:.2f}, r2_val={res['r2_val']:.4f}")
    print()
    print()
    print("-" * 140)

### Задание 5.2. Сравнение методов (0.75 балла)

Постройте график зависимости ошибки на обучающей выборке от номера итерации (все методы на одном графике).

Посмотрите на получившиеся результаты (таблички с метриками и график). Сравните методы между собой.

In [ ]:
best_params = {
    VanillaGradientDescent: {'lr': 0.280722, 'batch_size': None, 'color': 'blue', 'line_style': '-', 'marker': None},
    StochasticGradientDescent: {'lr': 0.085317, 'batch_size': 2048, 'color': 'red', 'line_style': '-', 'marker': None},
    SAGDescent: {'lr': 10.000000, 'batch_size': 4096, 'color': 'green', 'line_style': '-.', 'marker': None},
    MomentumDescent: {'lr': 0.417532, 'batch_size': None, 'color': 'orange', 'line_style': '-', 'marker': None},
    Adam: {'lr': 3.039195, 'batch_size': None, 'color': 'purple', 'line_style': '-', 'marker': None}
}

model_names = {
    VanillaGradientDescent: 'VanillaGradientDescent',
    StochasticGradientDescent: 'StochasticGradientDescent',
    SAGDescent: 'SAGDescent',
    MomentumDescent: 'MomentumDescent',
    Adam: 'Adam'
}

tolerance = 1e-5
max_iter = 1000

loss_histories = {}
final_metrics = {}

for model_class, params in best_params.items():
    
    if model_class in [StochasticGradientDescent, SAGDescent]:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            batch_size=params['batch_size'],
            lr_schedule=ConstantLR(params['lr'])
        )
    else:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            lr_schedule=ConstantLR(params['lr'])
        )
    
    model = CustomLinearRegression(optimizer=optimizer, loss_function=MSELoss())
    model.fit(X_train_np, y_train_np)
    
    loss_histories[model_names[model_class]] = model.loss_history
    
    train_loss = model.compute_loss(X_train_np, y_train_np)
    val_loss = model.compute_loss(X_val_np, y_val_np)
    test_loss = model.compute_loss(X_test_np, y_test_np)
    r2_test = r2_score(y_test_np, model.predict(X_test_np))
    
    final_metrics[model_names[model_class]] = {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'test_loss': test_loss,
        'r2_test': r2_test,
        'iterations': optimizer.iteration
    }

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
for method_name, history in loss_histories.items():
    iterations = range(1, min(201, len(history) + 1))
    color = best_params[[k for k, v in model_names.items() if v == method_name][0]]['color']
    line_style = best_params[[k for k, v in model_names.items() if v == method_name][0]]['line_style']
    
    ax1.plot(iterations, history[:200], 
            label=method_name,
            color=color,
            linestyle=line_style,
            linewidth=2,
            alpha=0.8)

ax1.set_xlabel('Номер итерации', fontsize=12)
ax1.set_ylabel('MSE', fontsize=12)
ax1.set_title('Первые 200 итераций', fontsize=14)
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3, linestyle='-')
ax1.set_xlim(0, 200)

ax2 = axes[1]
for method_name, history in loss_histories.items():
    iterations = range(1, len(history) + 1)
    color = best_params[[k for k, v in model_names.items() if v == method_name][0]]['color']
    line_style = best_params[[k for k, v in model_names.items() if v == method_name][0]]['line_style']
    
    ax2.plot(iterations, history, 
            label=method_name,
            color=color,
            linestyle=line_style,
            linewidth=2,
            alpha=0.8)

ax2.set_xlabel('Номер итерации', fontsize=12)
ax2.set_ylabel('MSE', fontsize=12)
ax2.set_title('Все 1000 итераций', fontsize=14)
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(True, alpha=0.3, linestyle='-')
ax2.set_xlim(0, max_iter)

plt.tight_layout()
plt.savefig('optimization_convergence.png', dpi=300, bbox_inches='tight')
plt.show()


print("\n" + "-" * 80)
print("Итоговые результаты")
print("-" * 80)

print(f"\n{'Метод':<25} | {'Best lambda':>10} | {'Train MSE':>12} | {'Val MSE':>12} | {'Test MSE':>12} | {'R² Test':>10} | {'Iterations':>10}")
print("-" * 110)

for method_name, metrics in final_metrics.items():
    lr = best_params[[k for k, v in model_names.items() if v == method_name][0]]['lr']
    print(f"{method_name:<25} | {lr:>10.6f} | "
          f"{metrics['train_loss']:>12.6f} | {metrics['val_loss']:>12.6f} | "
          f"{metrics['test_loss']:>12.6f} | {metrics['r2_test']:>10.6f} | {metrics['iterations']:>10}")

## Задание 6. Стохастический градиентный спуск и размер батча (1 балл)

В этом задании вам предстоит исследовать влияние размера батча на работу стохастического градиентного спуска.

* Сделайте по несколько запусков (например, $k = 10$) стохастического градиентного спуска на обучающей выборке для каждого размера батча из перебираемого списка. Замерьте время в секундах и количество итераций до сходимости. Посчитайте среднее этих значений для каждого размера батча.
* Постройте график зависимости количества шагов до сходимости от размера батча. _(под сходимостью понимается достижение критерия останова)_
* Постройте график зависимости времени до сходимости от размера батча.

Посмотрите на получившиеся результаты. Какие выводы можно сделать про подбор размера батча для стохастического градиентного спуска?

In [ ]:
import time

best_lr = 0.085317
batch_sizes = [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
k = 10
max_iter = 1000
tolerance = 1e-5

results = {
    'batch_size': [],
    'avg_iterations': [],
    'avg_time': []
}

for batch_size in batch_sizes:
    iterations_list = []
    time_list = []
    
    print(f"\n--- Batch size: {batch_size} ---")
    
    for run in range(k):
        optimizer = StochasticGradientDescent(
            lr_schedule=ConstantLR(best_lr),
            batch_size=batch_size,
            max_iter=max_iter,
            tolerance=tolerance
        )
        
        model = CustomLinearRegression(optimizer=optimizer)
        
        start_time = time.time()
        model.fit(X_train_np, y_train_np)
        end_time = time.time()
        
        elapsed_time = end_time - start_time
        iterations = optimizer.iteration
        
        iterations_list.append(iterations)
        time_list.append(elapsed_time)
    
    avg_iter = np.mean(iterations_list)
    avg_time = np.mean(time_list)
    
    results['batch_size'].append(batch_size)
    results['avg_iterations'].append(avg_iter)
    results['avg_time'].append(avg_time)
    
    print(f"  Average: iter={avg_iter:.1f}, time={avg_time:.4f}s")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(results['batch_size'], results['avg_iterations'], 'o-', linewidth=2)
plt.xlabel('Batch size')
plt.ylabel('Среднее количество итераций')
plt.title('Зависимость итераций от размера батча')
plt.grid(True, alpha=0.3)
plt.xscale('log')

plt.subplot(1, 2, 2)
plt.plot(results['batch_size'], results['avg_time'], 'o-', linewidth=2, color='red')
plt.xlabel('Batch size')
plt.ylabel('Среднее время до сходимости (сек)')
plt.title('Зависимость времени от размера батча')
plt.grid(True, alpha=0.3)
plt.xscale('log')

plt.tight_layout()
plt.show()

print("\n" + "-"*60)
print("Итоговые результаты")
print("-"*60)
print(f"{'Batch Size':<12} | {'Ср. итерации':<15} | {'Ср. время (с)':<15}")
print("-"*50)

for i in range(len(batch_sizes)):
    print(f"{results['batch_size'][i]:<12} | {results['avg_iterations'][i]:<15.1f} | {results['avg_time'][i]:<15.4f}")

**Выводы:**

Видно, что при малых размерах батча (32–1024) алгоритм не успевает сойтись за 1000 итераций, а время выполнения постепенно растёт. Оптимальным оказался размер батча 4096, при котором достигается минимальное время сходимости (2.22 секунды) при 305 итерациях. Дальнейшее увеличение батча до 8192 незначительно сокращает число итераций, но приводит к росту времени выполнения из-за вычислительных затрат

## Задание 7. Регуляризация (0.5 балла)

В этом задании вам предстоит исследовать влияние регуляризации на работу различных методов градиентного спуска. Напомним, регуляризация – это добавка к функции потерь, которая штрафует за норму весов. Мы будем использовать $L_2$-регуляризацию, таким образом функция потерь приобретает следующий вид:

$$
    Q(w) = \dfrac{1}{\ell} \sum\limits_{i=1}^{\ell} (a_w(x_i) - y_i)^2 + \dfrac{\mu}{2} \| w \|^2
$$

Допишите класс `linear_regression.L2Regularization`, следуя интерфейсу и докуметации.

Используя регуляризованный лосс в эксприментах, найдите лучшие параметры обучения с регуляризацией аналогично 5 заданию. Будем подбирать длину шага $\lambda$ (`lambda_`) и коэффициент регуляризации $\mu$ (`mu`).

Сравните для каждого метода результаты с регуляризацией и без регуляризации (нужно опять сохранить ошибку и качество по метрике $R^2$ на обучающей и тестовой выборках и количество итераций до сходимости).

Постройте для каждого метода график со значениями функции потерь MSE с регуляризацией и без регуляризации (всего должно получиться 5 графиков).

Посмотрите на получившиеся результаты. Какие можно сделать выводы, как регуляризация влияет на сходимость? Как изменилось качество на обучающей выборке? На тестовой? Чем вы можете объяснить это?

In [ ]:
from descents import (ConstantLR, VanillaGradientDescent, StochasticGradientDescent, SAGDescent, MomentumDescent, Adam)

from linear_regression import (MSELoss, L2Regularization, CustomLinearRegression)

lambda_grid = np.logspace(-4, 1, 30)
mu_grid = [0.0001, 0.0001, 0.001, 0.01, 0.1, 1.0]

best_params_reg = {}
best_results_reg = {}

model_names = {VanillaGradientDescent: 'VanillaGradientDescent', StochasticGradientDescent: 'StochasticGradientDescent', SAGDescent: 'SAGDescent', MomentumDescent: 'MomentumDescent', Adam: 'Adam'}

tolerance = 1e-5
max_iter = 1000

batch_size = 4096
best_result_sagd_reg = {
    "best_lr": None,
    "best_mu": None,
    "iterations": None,
    "train_loss": np.inf,
    "val_loss": np.inf,
    "r2_val": None,
    "batch_size": batch_size
}

min_loss = 1e9

for lr in lambda_grid:
    for mu in mu_grid:
        optimizer = SAGDescent(
            tolerance=tolerance,
            max_iter=max_iter,
            batch_size=batch_size,
            lr_schedule=ConstantLR(lr)
        )
        
        loss_fn = L2Regularization(MSELoss(), mu_rate=mu)
        model = CustomLinearRegression(optimizer=optimizer, loss_function=loss_fn)
        model.fit(X_train_np, y_train_np)
        
        train_loss = model.compute_loss(X_train_np, y_train_np)
        val_loss = model.compute_loss(X_val_np, y_val_np)
        
        if not (np.isfinite(train_loss) and np.isfinite(val_loss)):
            continue
        
        r2_val = r2_score(y_val_np, model.predict(X_val_np))
        iterations = optimizer.iteration
        
        if val_loss < min_loss:
            min_loss = val_loss
            best_result_sagd_reg.update({
                "best_lr": lr,
                "best_mu": mu,
                "iterations": iterations,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "r2_val": r2_val,
                "batch_size": batch_size
            })

print("\n" + "-" * 50)
print(f"BEST for SAGDescent with reg: lr={best_result_sagd_reg['best_lr']:.6f}, mu={best_result_sagd_reg['best_mu']:.4f}")
print(f"val_mse={best_result_sagd_reg['val_loss']:.6f}, r2_val={best_result_sagd_reg['r2_val']:.6f}")
print("-" * 50)

model_name_no = {VanillaGradientDescent: "VanillaGradientDescent", StochasticGradientDescent: "StochasticGradientDescent", MomentumDescent: "MomentumDescent", Adam: "Adam"}

for model_class in model_name_no:
    method_name = model_name_no[model_class]
    
    min_loss = 1e9
    best_result_reg = {
        "best_lr": None,
        "best_mu": None,
        "iterations": None,
        "train_loss": np.inf,
        "val_loss": np.inf,
        "r2_val": None
    }
    
    for lr in lambda_grid:
        for mu in mu_grid:
            optimizer = model_class(
                tolerance=tolerance,
                max_iter=max_iter,
                lr_schedule=ConstantLR(lr)
            )
            
            loss_fn = L2Regularization(MSELoss(), mu_rate=mu)
            model = CustomLinearRegression(optimizer=optimizer, loss_function=loss_fn)
            model.fit(X_train_np, y_train_np)
            
            train_loss = model.compute_loss(X_train_np, y_train_np)
            val_loss = model.compute_loss(X_val_np, y_val_np)
            
            if not (np.isfinite(train_loss) and np.isfinite(val_loss)):
                continue
            
            r2_val = r2_score(y_val_np, model.predict(X_val_np))
            iterations = optimizer.iteration
            
            
            if val_loss < min_loss:
                min_loss = val_loss
                best_result_reg.update({
                    "best_lr": lr,
                    "best_mu": mu,
                    "iterations": iterations,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "r2_val": r2_val
                })
    
    print("\n" + "-" * 50)
    print(f"BEST for {method_name} with reg: lr={best_result_reg['best_lr']:.6f}, mu={best_result_reg['best_mu']:.4f}")
    print(f"val_mse={best_result_reg['val_loss']:.6f}, r2_val={best_result_reg['r2_val']:.6f}")
    print("-" * 50)
    
    best_results_reg[method_name] = best_result_reg

best_results_reg['SAGDescent'] = best_result_sagd_reg

In [ ]:
print(f"SAGDescent: {best_result_sagd_reg['iterations']} итераций")

for method_name, results in best_results_reg.items():
    if method_name != 'SAGDescent':
        print(f"{method_name}: {results['iterations']} итераций")

In [ ]:
from descents import (
    ConstantLR,
    VanillaGradientDescent,
    StochasticGradientDescent,
    SAGDescent,
    MomentumDescent,
    Adam,
)
from linear_regression import (
    MSELoss,
    L2Regularization,
    CustomLinearRegression,
)

best_params = {
    VanillaGradientDescent: {'lr': 0.280722, 'batch_size': None, 'color': 'blue', 'line_style': '-', 'marker': None},
    StochasticGradientDescent: {'lr': 0.085317, 'batch_size': 2048, 'color': 'red', 'line_style': '-', 'marker': None},
    SAGDescent: {'lr': 10.000000, 'batch_size': 4096, 'color': 'green', 'line_style': '-.', 'marker': None},
    MomentumDescent: {'lr': 0.417532, 'batch_size': None, 'color': 'orange', 'line_style': '-', 'marker': None},
    Adam: {'lr': 3.039195, 'batch_size': None, 'color': 'purple', 'line_style': '-', 'marker': None}
}

best_params_reg = {
    VanillaGradientDescent: {'lr': 0.280722, 'mu': 0.0001, 'batch_size': None, 'color': 'blue', 'line_style': '-'},
    StochasticGradientDescent: {'lr': 0.126896, 'mu': 0.0001, 'batch_size': 2048, 'color': 'red', 'line_style': '-'},
    SAGDescent: {'lr': 10.000000, 'mu': 0.0001, 'batch_size': 4096, 'color': 'green', 'line_style': '-.'},
    MomentumDescent: {'lr': 0.417532, 'mu': 0.0001, 'batch_size': None, 'color': 'orange', 'line_style': '-'},
    Adam: {'lr': 3.039195, 'mu': 0.0001, 'batch_size': None, 'color': 'purple', 'line_style': '-'}
}

model_names = {
    VanillaGradientDescent: 'VanillaGradientDescent',
    StochasticGradientDescent: 'StochasticGradientDescent',
    SAGDescent: 'SAGDescent',
    MomentumDescent: 'MomentumDescent',
    Adam: 'Adam'
}

tolerance = 1e-5
max_iter = 1000

loss_histories_no_reg = {}

for model_class, params in best_params.items():
    method_name = model_names[model_class]
    
    if model_class in [StochasticGradientDescent, SAGDescent]:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            batch_size=params['batch_size'],
            lr_schedule=ConstantLR(params['lr'])
        )
    else:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            lr_schedule=ConstantLR(params['lr'])
        )
    
    model = CustomLinearRegression(optimizer=optimizer, loss_function=MSELoss())
    model.fit(X_train_np, y_train_np)
    loss_histories_no_reg[method_name] = model.loss_history

loss_histories_reg = {}

for model_class, params in best_params_reg.items():
    method_name = model_names[model_class]
    
    if model_class in [StochasticGradientDescent, SAGDescent]:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            batch_size=params['batch_size'],
            lr_schedule=ConstantLR(params['lr'])
        )
    else:
        optimizer = model_class(
            tolerance=tolerance,
            max_iter=max_iter,
            lr_schedule=ConstantLR(params['lr'])
        )
    
    loss_fn = L2Regularization(MSELoss(), mu_rate=params['mu'])
    model = CustomLinearRegression(optimizer=optimizer, loss_function=loss_fn)
    model.fit(X_train_np, y_train_np)
    loss_histories_reg[method_name] = model.loss_history

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (method_name, history_reg) in enumerate(loss_histories_reg.items()):
    ax = axes[idx]
    
    history_no_reg = loss_histories_no_reg[method_name]
    
    min_len = min(len(history_no_reg), len(history_reg))
    iterations = range(1, min_len + 1)
    
    color_no_reg = best_params[[k for k, v in model_names.items() if v == method_name][0]]['color']
    ax.plot(iterations, history_no_reg[:min_len], 
            label=f'{method_name} (без регуляризации)',
            color='orange',
            linestyle='-',
            linewidth=2,
            alpha=0.7)
    
    mu_value = best_params_reg[list(best_params_reg.keys())[idx]]['mu']
    ax.plot(iterations, history_reg[:min_len], 
            label=f'{method_name} (μ={mu_value:.4f})',
            color='pink',
            linestyle='--',
            linewidth=2,
            alpha=0.9)
    
    ax.set_xlabel('Номер итерации', fontsize=11)
    ax.set_ylabel('MSE', fontsize=11)
    ax.set_title(f'{method_name}', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    ax.set_xlim(0, min_len)

axes[5].set_visible(False)

plt.suptitle('Сравнение сходимости с регуляризацией и без', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('regularization_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

**Вывод:**

Анализ показал, что регуляризация с коэффициентом μ = 0.0001 практически не повлияла на сходимость методов — графики почти совпадают, так как коэффициент слишком мал. На обучающей выборке ошибка немного выросла, что ожидаемо, а на тестовой — чуть снизилась, то есть качество предсказаний улучшилось. Это объясняется тем, что даже слабая регуляризация немного ограничивает веса, не давая модели слишком подстраиваться под шум в данных, и она начинает чуть лучше обобщать на новых примерах.

## Задание 8. Альтернативные функции потерь (1 балл)

В этом задании вам предстоит использовать другую функцию потерь для нашей задачи регрессии. В качестве функции потерь мы выбрали **LogCosh** и **HuberLoss**:

$$
    L(y, a)
    =
    \log\left(\cosh(a - y)\right).
$$

$$
L_{\text{Huber}}(y, a) = \frac{1}{n} \sum_{i=1}^{n}
\begin{cases}
   \frac{1}{2} (a_i - y_i)^2, & \text{если } |a_i - y_i| < \delta, \\
   \delta \cdot |a_i - y_i| - \frac{1}{2} \delta^2, & \text{если } |a_i - y_i| \geq \delta,
\end{cases}
$$

Самостоятельно продифференцируйте данные функции потерь чтобы найти их градиенты _(требуется показать не только результат, но и промежуточные вычисления)_:

**Решение:**

Дифференцирование функции потерь $L(y, a) = log(cosh(a-y))$:

Рассмотрим $log(cosh(r))$:

$\log(\cosh(r))' = \frac{\sinh(r)}{\cosh(r)} = tgh(r)$

Получаем $\nabla_wL(w) = \frac{1}{n}\sum_{i = 1}^n L'(r_i) \cdot \nabla_wr_i$

$r_i = x^T_i \cdot x_i - y_i$

$\nabla_wr_i = x_i$

$\nabla_wL(w) = \frac{1}{n}\sum_{i = 1}^nx_i \cdot tgh(r_i)$

$\nabla_wL(w) = \frac{1}{n} X^T \cdot tgh(Xw - y)$

Дифференцирование функции потерь $L_{\text{Huber}}(y, a) = \frac{1}{n} \sum_{i=1}^{n}
\begin{cases}
   \frac{1}{2} (a_i - y_i)^2, & \text{если } |a_i - y_i| < \delta, \\
   \delta \cdot |a_i - y_i| - \frac{1}{2} \delta^2, & \text{если } |a_i - y_i| \geq \delta,
\end{cases}$:

Сразу рассмотрим на r:

$L_{\text{Huber}}(r) = \frac{1}{n} \sum_{i=1}^{n}
\begin{cases}
   \frac{1}{2} (r)^2, & \text{если } |r| < \delta, \\
   \delta \cdot |r| - \frac{1}{2} \delta^2, & \text{если } |r| \geq \delta,
\end{cases}$

Разобьем на 2 случая:

Случай 1 $|r| < \delta$:

$L_H = \frac{1}{2}r^2$, следовательно $L'_H = r$

Случай 2 $|r| \ge \delta$:

$L_H = \delta |r| - \frac{1}{2}\delta^2$, следовательно $L'_H = \delta \cdot \frac{d}{dr}|r| = \delta sign(r)$

Значит $L'_H(r_i) = \begin{cases}
    r_i, |r_i| < \delta \\
    \delta \cdot sign(r_i), |r_i| \ge \delta
\end{cases}$

Обозначу $L'_H(r_i) = h_i$, тогда:

$\nabla L(w) = \frac{1}{n}X^Th$

Возьмем $clip(r, -\delta, \delta)$= 
\begin{cases}
   -\delta, & r_i < -\delta \\
    r_i, & -\delta \leq r_i \leq \delta \\
    \delta, & r_i > \delta
\end{cases}

Итог: $\nabla_wL_h(w) = \frac{1}{n}X^T\cdot clip(X_{w-y}, -\delta, \delta)$

Программно реализуйте функции потерь и их градиенты для LogCosh и HuberLoss в файле `linear_regression.py`. После этого обучите все пять методов градиентного спуска (без регуляризации) с этими лоссами аналогично заданию 5 и сравните качество с результатами из задания 5, где использовался MSE.

Имплементировать эти функции потерь необходимо при помощи наследования от `linear_regression.LossFunction` и имплементирования всех абстрактных методов. Аналитическое решение для этих функций выводить и имплементировать не требуется.



In [ ]:
from descents import (ConstantLR, VanillaGradientDescent, StochasticGradientDescent, SAGDescent, MomentumDescent, Adam)
from linear_regression import (LogCoshLoss, HuberLoss, CustomLinearRegression)

lambda_grid = np.logspace(-4, 1, 30)
delta_grid = [0.5, 1.0, 2.0, 5.0]
tolerance = 1e-5
max_iter = 1000
batch_size = 4096

model_names = {VanillaGradientDescent: 'VanillaGradientDescent', StochasticGradientDescent: 'StochasticGradientDescent', SAGDescent: 'SAGDescent', MomentumDescent: 'MomentumDescent', Adam: 'Adam'}

print("\n" + "-"*70)
print("Результаты для LogCoshLoss")
print("-"*70)

best_results_logcosh = {}

for model_class in [VanillaGradientDescent, StochasticGradientDescent, MomentumDescent, Adam, SAGDescent]:
    method_name = model_names[model_class]
    min_loss = 1e9
    best_result = {
        "best_lr": None,
        "iterations": None,
        "train_loss": np.inf,
        "val_loss": np.inf,
        "r2_val": None
    }
    
    for lr in lambda_grid:
        if model_class == SAGDescent:
            optimizer = model_class(
                tolerance=tolerance,
                max_iter=max_iter,
                batch_size=batch_size,
                lr_schedule=ConstantLR(lr)
            )
        elif model_class == StochasticGradientDescent:
            optimizer = model_class(
                tolerance=tolerance,
                max_iter=max_iter,
                batch_size=2048,
                lr_schedule=ConstantLR(lr)
            )
        else:
            optimizer = model_class(
                tolerance=tolerance,
                max_iter=max_iter,
                lr_schedule=ConstantLR(lr)
            )
        
        model = CustomLinearRegression(optimizer=optimizer, loss_function=LogCoshLoss())
        model.fit(X_train_np, y_train_np)
        
        val_loss = model.compute_loss(X_val_np, y_val_np)
        
        if not np.isfinite(val_loss):
            continue
        
        r2_val = r2_score(y_val_np, model.predict(X_val_np))
        
        if val_loss < min_loss:
            min_loss = val_loss
            best_result.update({
                "best_lr": lr,
                "iterations": optimizer.iteration,
                "train_loss": model.compute_loss(X_train_np, y_train_np),
                "val_loss": val_loss,
                "r2_val": r2_val
            })
    
    best_results_logcosh[method_name] = best_result
    print(f"{method_name:<25} | lr={best_result['best_lr']:.6f} | val_loss={best_result['val_loss']:.6f} | R²={best_result['r2_val']:.6f}")

print("\n" + "-"*70)
print("Результаты для HuberLoss")
print("-"*70)

for delta in delta_grid:
    print(f"\n--- delta = {delta} ---")
    
    for model_class in [VanillaGradientDescent, StochasticGradientDescent, MomentumDescent, Adam, SAGDescent]:
        method_name = model_names[model_class]
        min_loss = 1e9
        best_result = {
            "best_lr": None,
            "iterations": None,
            "train_loss": np.inf,
            "val_loss": np.inf,
            "r2_val": None
        }
        
        for lr in lambda_grid:
            if model_class == SAGDescent:
                optimizer = model_class(
                    tolerance=tolerance,
                    max_iter=max_iter,
                    batch_size=batch_size,
                    lr_schedule=ConstantLR(lr)
                )
            elif model_class == StochasticGradientDescent:
                optimizer = model_class(
                    tolerance=tolerance,
                    max_iter=max_iter,
                    batch_size=2048,
                    lr_schedule=ConstantLR(lr)
                )
            else:
                optimizer = model_class(
                    tolerance=tolerance,
                    max_iter=max_iter,
                    lr_schedule=ConstantLR(lr)
                )
            
            model = CustomLinearRegression(optimizer=optimizer, loss_function=HuberLoss(delta=delta))
            model.fit(X_train_np, y_train_np)
            
            val_loss = model.compute_loss(X_val_np, y_val_np)
            
            if not np.isfinite(val_loss):
                continue
            
            r2_val = r2_score(y_val_np, model.predict(X_val_np))
            
            if val_loss < min_loss:
                min_loss = val_loss
                best_result.update({
                    "best_lr": lr,
                    "iterations": optimizer.iteration,
                    "train_loss": model.compute_loss(X_train_np, y_train_np),
                    "val_loss": val_loss,
                    "r2_val": r2_val
                })
        
        print(f"{method_name:<25} | lr={best_result['best_lr']:.6f} | val_loss={best_result['val_loss']:.6f} | R²={best_result['r2_val']:.6f}")

Сравнение результатов показывает, что обе альтернативные функции потерь — LogCosh и Huber — работают заметно лучше, чем обычный MSE. Значения val_loss для новых лоссов находятся в районе 0.07–0.09, в то время как для MSE они были около 0.17, то есть ошибка снизилась примерно в два раза. При этом метрика R² тоже подросла — с 0.85 до 0.855 у некоторых методов, что говорит о более точных предсказаниях.

Среди всех вариантов лучше всего себя показал HuberLoss с дельтой 0.5 — у него самые низкие ошибки на валидации, особенно у MomentumDescent и Adam. LogCosh тоже неплох, но немного уступает Huber. При увеличении дельты ошибка начинает расти и приближаться к MSE, что логично, потому что Huber с большой дельтой ведёт себя почти как обычный квадрат ошибки.

Ещё можно заметить, что для новых функций потерь оптимальные значения learning rate часто оказываются больше (0.62, 0.92, даже 10 у SAG), чем для MSE (0.28, 0.41), видимо, потому что градиенты менее агрессивные и можно брать шаг побольше.

В целом, LogCosh и HuberLoss оказались более устойчивыми и дали лучшее качество, чем стандартный MSE